# `langchain-diffbot` quickstart

This notebook walks through the `langchain-diffbot` package end-to-end:

1. Install + authenticate
2. Run a basic Diffbot Knowledge Graph (KG) retrieval with a [DQL](https://docs.diffbot.com/reference/dql-quickstart) query
3. Shape retrieval output (`fields`, `content_fields`, `document_mapper`) so KG entities don't blow past LLM input-token limits
4. Wire the retriever into a Claude-powered research agent with `create_agent`

You'll need:

- A **Diffbot API token** — [app.diffbot.com/get-started](https://app.diffbot.com/get-started/)
- An **Anthropic API key** (for the agent section only) — [console.anthropic.com](https://console.anthropic.com/)

## 1. Install

In [ ]:
%pip install --quiet langchain-diffbot langchain langchain-anthropic python-dotenv

## 2. Authenticate

Put your keys in a `.env` next to this notebook, or paste them inline below. `getpass` keeps them out of the notebook output.

In [ ]:
import getpass
import os

from dotenv import load_dotenv

load_dotenv()

if not os.getenv("DIFFBOT_API_TOKEN"):
    os.environ["DIFFBOT_API_TOKEN"] = getpass.getpass("DIFFBOT_API_TOKEN: ")

if not os.getenv("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("ANTHROPIC_API_KEY: ")

## 3. Basic retrieval

`DiffbotKnowledgeGraphRetriever` is a standard LangChain `BaseRetriever`. The `query` you pass to `.invoke()` is a [DQL](https://docs.diffbot.com/reference/dql-quickstart) expression — not natural language. A few patterns you'll use a lot:

| What you want | DQL |
| --- | --- |
| Filter by entity type | `type:Organization`, `type:Person`, `type:Article` |
| Exact match | `name:"Diffbot"` |
| Nested fields | `location.city.name:"Austin"` |
| AND (combine with spaces) | `type:Organization industries:"Robotics"` |
| Sort | `sortBy:nbEmployees desc` |

In [ ]:
from langchain_diffbot import DiffbotKnowledgeGraphRetriever

retriever = DiffbotKnowledgeGraphRetriever(k=5)

docs = retriever.invoke(
    'type:Organization industries:"Artificial Intelligence" location.city.name:"Boston"'
)

for d in docs:
    print(d.metadata.get("name"), "—", d.page_content[:120])

## 4. Shaping the output

Diffbot KG entities can be huge — nested `suppliers`, `employments`, `articles`, `tags`, etc. One unshaped `invoke()` can drop tens of thousands of tokens into a prompt and blow past per-minute input-token limits. The retriever gives you three knobs:

1. `fields=[...]` — allowlist of top-level metadata keys to keep
2. `content_fields=[...]` — ordered priority for which field becomes `page_content` (first non-empty value wins)
3. `document_mapper=fn` — full override; ignores the other two

### 4a. Field projection (recommended for agent / tool-use)

In [ ]:
retriever = DiffbotKnowledgeGraphRetriever(
    k=3,
    fields=["id", "type", "name", "homepageUri", "nbEmployees", "industries"],
)

docs = retriever.invoke('type:Organization location.city.name:"Austin" industries:"Robotics"')

for d in docs:
    print(d.metadata)
    print("  content:", d.page_content[:200], "\n")

### 4b. Pick which field becomes `page_content`

In [ ]:
retriever = DiffbotKnowledgeGraphRetriever(
    k=3,
    fields=["id", "name"],
    content_fields=["summary", "description", "name"],
)

for d in retriever.invoke('type:Organization name:"Anthropic"'):
    print(d.metadata["name"], "—", d.page_content[:160])

### 4c. Full control with `document_mapper`

Use this when you need a custom Document shape — nested-field projection, formatted strings, derived metadata, etc.

In [ ]:
from langchain_core.documents import Document


def mapper(entity: dict) -> Document:
    city = (entity.get("location") or {}).get("city", {}).get("name")
    return Document(
        page_content=entity.get("summary") or entity.get("description", ""),
        metadata={
            "id": entity.get("id"),
            "name": entity.get("name"),
            "city": city,
            "employees": entity.get("nbEmployees"),
        },
    )


retriever = DiffbotKnowledgeGraphRetriever(k=3, document_mapper=mapper)

for d in retriever.invoke('type:Organization industries:"Biotechnology" sortBy:nbEmployees desc'):
    print(d.metadata, "—", d.page_content[:120])

## 5. Async

Both surfaces are native — `ainvoke` runs on a real `httpx.AsyncClient`, not a thread-pool fallback. This matters when you fan out many KG queries from one event loop.

In [ ]:
import asyncio

retriever = DiffbotKnowledgeGraphRetriever(k=3, fields=["id", "name", "industries"])

queries = [
    'type:Organization location.city.name:"Austin" industries:"Robotics"',
    'type:Organization location.city.name:"Boston" industries:"Biotechnology"',
    'type:Organization location.city.name:"San Francisco" industries:"Artificial Intelligence"',
]

results = await asyncio.gather(*(retriever.ainvoke(q) for q in queries))

for q, docs in zip(queries, results, strict=True):
    print(q)
    for d in docs:
        print("  •", d.metadata.get("name"))

## 6. Build a research agent

Wrap the retriever in a `@tool` and hand it to `create_agent` so Claude can generate DQL, call the KG, and refine if needed. This mirrors the [`langchain-diffbot-demo`](https://github.com/shixish/langchain-diffbot) company-research CLI.

In [ ]:
from functools import lru_cache

import httpx
from langchain.agents import create_agent
from langchain_core.tools import tool

_KG_FIELDS = [
    "id",
    "type",
    "name",
    "homepageUri",
    "nbEmployees",
    "industries",
    "location",
    "employments",
    "date",
]


@lru_cache(maxsize=1)
def _get_retriever() -> DiffbotKnowledgeGraphRetriever:
    return DiffbotKnowledgeGraphRetriever(k=5, fields=_KG_FIELDS)


@tool
def search_kg(dql_query: str) -> list[dict]:
    """Search the Diffbot Knowledge Graph with a DQL query.

    DQL syntax:
        - Filter by type: `type:Organization`, `type:Person`, `type:Article`
        - Exact match: `name:"Diffbot"`
        - Nested fields: `location.city.name:"Austin"`
        - AND with spaces: `type:Organization industries:"Robotics"`
        - Sort: `sortBy:nbEmployees desc`

    Returns a list of entity dicts with `summary`, `id`, `type`, `name`, and a
    handful of projected fields. Refine the query if you need other fields.
    """
    try:
        docs = _get_retriever().invoke(dql_query)
    except httpx.HTTPStatusError as exc:
        return [{"error": f"Diffbot rejected the query ({exc.response.status_code}). Refine the DQL."}]
    return [{"summary": d.page_content, **d.metadata} for d in docs]


SYSTEM_PROMPT = """\
You are a company-research assistant with access to the Diffbot Knowledge Graph.

Use the `search_kg` tool to look up Organizations, People, Articles, and Places.
The tool takes a DQL query. Prefer specific, filterable DQL over broad queries.

Iterate when useful: if a query returns no results, loosen it; if too many,
tighten it (add filters, restrict by location/industry, change sort order).

When you answer, cite the entity IDs you used (e.g. "(Diffbot, id=E1234)").
Keep answers concise and factual. If the KG does not contain the information,
say so plainly rather than guessing.
"""

agent = create_agent(
    model="anthropic:claude-sonnet-4-6",
    tools=[search_kg],
    system_prompt=SYSTEM_PROMPT,
)

Ask it a research question. The agent will choose its own DQL, may iterate, and answers with citations.

In [ ]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What companies in Austin work on robotics?"}]}
)

print(result["messages"][-1].content)

Inspect the full trace — tool calls + tool outputs — to see the DQL the agent ran:

In [ ]:
for m in result["messages"]:
    print(f"[{m.type}]", getattr(m, "content", "") or getattr(m, "tool_calls", ""))

## Where to go next

- [DQL reference](https://docs.diffbot.com/reference/dql-quickstart) — full query language
- [Diffbot KG entity schema](https://docs.diffbot.com/reference/knowledge-graph-overview) — what fields exist on each entity type
- [`langchain-diffbot` README](https://github.com/shixish/langchain-diffbot) — reference docs for the retriever
- [`create_agent` docs](https://docs.langchain.com/oss/python/langchain/agents) — customize the agent loop (memory, structured output, middleware)